# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 3: HUẤN LUYỆN MÔ HÌNH, ĐÁNH GIÁ & TRIỂN KHAI (MODELING, EVALUATION & DEPLOYMENT)

```text
[train_test_data & preprocessor] ──> Mục 6 (Huấn luyện 3 mô hình: Naive Bayes, Logistic, Linear SVM)
                                  ──> Mục 7 (Đánh giá kiểm thử, Confusion Matrix, Error Analysis & Feature Importance)
                                  ──> [best_model.joblib & metadata.json] ──> Triển khai (app.py Streamlit & CLI)
```


In [ ]:
# Khai báo thư viện & Nạp dữ liệu đã xử lý từ Notebook 2
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', palette='muted')

DATA_INTERMEDIATE_PATH = '../data/processed/train_test_data.joblib'
if not os.path.exists(DATA_INTERMEDIATE_PATH):
    raise FileNotFoundError(f'❌ Chưa tìm thấy {DATA_INTERMEDIATE_PATH}. Bạn cần chạy hoàn tất Notebook 2 trước!')

data_dict = joblib.load(DATA_INTERMEDIATE_PATH)
X_train, X_test = data_dict['X_train'], data_dict['X_test']
y_train, y_test = data_dict['y_train'], data_dict['y_test']
preprocessor = joblib.load('../models/preprocessor.joblib')
print(f'✅ Đã nạp thành công dữ liệu từ Notebook 2: Train={len(X_train):,} mẫu | Test={len(X_test):,} mẫu')

---
## Mục 6: Huấn luyện mô hình — Naive Bayes, Logistic Regression & Linear SVM


In [ ]:
# Viết code khởi tạo 3 Pipeline hoàn chỉnh (preprocessor + model) và thực hiện 5-Fold Stratified Cross Validation trên tập Train tại đây
# -----------------------------------------------------------------------------
# 1. KHỞI TẠO CÁC MÔ HÌNH VÀ 3 PIPELINE CỐT LÕI
# -----------------------------------------------------------------------------
# LinearSVC kết hợp Platt Scaling qua CalibratedClassifierCV để sinh xác suất dự đoán
linear_svc_base = LinearSVC(random_state=RANDOM_SEED, max_iter=3000, dual='auto')
calibrated_linear_svc = CalibratedClassifierCV(estimator=linear_svc_base, method='sigmoid', cv=3)

# 3 Pipeline phân loại hoàn chỉnh kết hợp với preprocessor từ Notebook 2
pipelines = {
    "Multinomial Naive Bayes": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MultinomialNB())
    ]),
    "Logistic Regression (L1)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            penalty='l1', 
            solver='liblinear', 
            random_state=RANDOM_SEED, 
            max_iter=1000
        ))
    ]),
    "Linear SVM (Platt Scaling)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", calibrated_linear_svc)
    ])
}

# -----------------------------------------------------------------------------
# 2. ĐÁNH GIÁ CHÉO PHÂN TẦNG (STRATIFIED 5-FOLD CV) TRÊN TẬP TRAIN
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_summary = []

print("=" * 88)
print(f"{'MÔ HÌNH':<28} | {'F1-SCORE (Mean ± Std)':<22} | {'ROC-AUC':<12} | {'ACCURACY':<10}")
print("=" * 88)

for name, pipe in pipelines.items():
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    acc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    rec_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='recall', n_jobs=-1)
    prec_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='precision', n_jobs=-1)
    roc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    cv_summary.append({
        "Model": name,
        "Accuracy_Mean": acc_scores.mean(),
        "Accuracy_Std": acc_scores.std(),
        "Precision_Mean": prec_scores.mean(),
        "Recall_Mean": rec_scores.mean(),
        "F1_Mean": cv_scores.mean(),
        "F1_Std": cv_scores.std(),
        "ROC_AUC_Mean": roc_scores.mean(),
        
        "ROC_AUC_Std": roc_scores.std()
    })
    
    print(f"{name:<28} | {cv_scores.mean():.4f} ± {cv_scores.std():.4f}       | {roc_scores.mean():.4f}       | {acc_scores.mean():.4f}")

print("=" * 88)

# Xuất bảng tổng hợp kết quả kiểm định chéo
df_cv_summary = pd.DataFrame(cv_summary)
display(df_cv_summary)

In [ ]:
# Viết code tinh chỉnh siêu tham số bằng GridSearchCV cho cả Preprocessor (số từ TF-IDF, k của Chi-Square) và Mô hình tại đây
# -----------------------------------------------------------------------------
# 1. THIẾT LẬP LƯỚI THAM SỐ VÀ GRIDSEARCHCV CHO CẢ 3 MÔ HÌNH
# -----------------------------------------------------------------------------
tuned_models = {}

# Tham số chung cho bộ tiền xử lý (TF-IDF & Chi-Square SelectKBest)
preprocessor_param_grid = {
    'preprocessor__tfidf__max_features': [2000, 3000, 5000],
    'preprocessor__select__k': [800, 1000, 1500]
}

# 1.1. GridSearchCV cho Multinomial Naive Bayes
param_grid_nb = {
    **preprocessor_param_grid,
    'classifier__alpha': [0.01, 0.1, 0.5, 1.0]
}

print("⚙️ [1/3] Đang tinh chỉnh siêu tham số cho Multinomial Naive Bayes...")
grid_nb = GridSearchCV(
    estimator=pipelines["Multinomial Naive Bayes"],
    param_grid=param_grid_nb,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_nb.fit(X_train, y_train)
tuned_models["naive_bayes"] = grid_nb.best_estimator_
print(f"   👉 Best Params: {grid_nb.best_params_}")
print(f"   👉 Best CV F1 : {grid_nb.best_score_:.4f}\n")

# 1.2. GridSearchCV cho Logistic Regression (L1 Regularization)
param_grid_lr = {
    **preprocessor_param_grid,
    'classifier__C': [0.1, 1.0, 5.0, 10.0]
}

print("⚙️ [2/3] Đang tinh chỉnh siêu tham số cho Logistic Regression (L1)...")
grid_lr = GridSearchCV(
    estimator=pipelines["Logistic Regression (L1)"],
    param_grid=param_grid_lr,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_lr.fit(X_train, y_train)
tuned_models["logistic_regression"] = grid_lr.best_estimator_
print(f"   👉 Best Params: {grid_lr.best_params_}")
print(f"   👉 Best CV F1 : {grid_lr.best_score_:.4f}\n")

# 1.3. GridSearchCV cho Linear SVM + Platt Scaling
# Với CalibratedClassifierCV, tham số C của LinearSVC được truy cập qua: estimator__C
param_grid_svm = {
    **preprocessor_param_grid,
    'classifier__estimator__C': [0.05, 0.1, 0.5, 1.0, 2.0]
}

print("⚙️ [3/3] Đang tinh chỉnh siêu tham số cho Linear SVM (Platt Scaling)...")
grid_svm = GridSearchCV(
    estimator=pipelines["Linear SVM (Platt Scaling)"],
    param_grid=param_grid_svm,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_svm.fit(X_train, y_train)
tuned_models["linear_svm"] = grid_svm.best_estimator_
print(f"   👉 Best Params: {grid_svm.best_params_}")
print(f"   👉 Best CV F1 : {grid_svm.best_score_:.4f}\n")

# -----------------------------------------------------------------------------
# 2. XUẤT 3 MÔ HÌNH HOÀN CHỈNH VÀO THƯ MỤC ../models/
# -----------------------------------------------------------------------------
os.makedirs('../models', exist_ok=True)
print("=" * 60)
print("💾 LƯU TRỮ CÁC MÔ HÌNH ĐÃ HUẤN LUYỆN:")
print("=" * 60)

for filename, model_pipeline in tuned_models.items():
    save_path = f"../models/{filename}.joblib"
    joblib.dump(model_pipeline, save_path)
    print(f"  ✓ Đã lưu thành công: {save_path}")

print("=" * 60)

---
## Mục 7: Đánh giá so sánh 3 mô hình, Phân tích lỗi & Triển khai Demo


In [ ]:
# Viết code dự đoán trên Test set, tính toán Accuracy, Precision, Recall, F1-score và lập bảng so sánh hiệu năng tổng kết tại đây


In [ ]:
# Viết code trực quan hóa Ma trận nhầm lẫn (Confusion Matrix) cho cả 3 mô hình tại đây


In [ ]:
# Kỹ thuật Cô Tiên — Viết code trích xuất và vẽ Biểu đồ thanh ngang Top 15 từ khóa có trọng số báo hiệu Spam mạnh nhất tại đây


In [ ]:
# Viết code lưu mô hình tốt nhất vào '../models/best_model.joblib', xuất '../models/metadata.json' và viết hàm predict_spam(email_text) tại đây
